# 08 — Volatility Model, Probabilistic Forecasting & Risk Engine

Phase 8 of the Indian Equity AI project plan.

For every stock/day, produce:
1. A probability of next-day up-move (Phase 5's classifier, retrained here on the same split)
2. An expected-volatility forecast (a new, separate regression model)
3. A probabilistic next-day price range (10th/50th/90th percentile, via quantile regression)
4. A Low/Medium/High risk label (volatility + beta + historical drawdown + VaR)

"Every prediction ships with a probability, a range, and a risk label -- not just a point estimate."

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor

from src.ingestion.market_data import download_index_ohlcv
from src.preprocessing.clean import clean_ohlcv
from src.features.returns import add_returns as add_universe_returns
from src.features.risk import beta as compute_beta, max_drawdown as compute_max_drawdown
from src.models.dataset import FEATURE_COLUMNS, add_next_day_return, add_target, build_feature_table, chronological_split
from src.models.evaluate import classification_metrics
from src.risk.volatility_target import add_forward_volatility_target
from src.risk.quantiles import fit_quantile_models, predict_quantiles, return_quantiles_to_price_range
from src.risk.var_es import rolling_var_es
from src.risk.engine import compute_risk_score, risk_label_relative

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42
VOL_HORIZON = 5


def make_lgbm_classifier():
    return LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    )


def make_lgbm_regressor(objective="regression", alpha=None):
    kwargs = dict(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1, objective=objective,
    )
    if alpha is not None:
        kwargs["alpha"] = alpha
    return LGBMRegressor(**kwargs)

## Step 1: Build the base feature table + all three targets

In [2]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")

df = build_feature_table(technical)
df = add_target(df)  # next_day_direction (classification)
df = add_next_day_return(df)  # next_day_return (quantile regression)
df = add_forward_volatility_target(df, horizon=VOL_HORIZON)  # forward_volatility_5d (regression)

vol_target_col = f"forward_volatility_{VOL_HORIZON}d"
required = FEATURE_COLUMNS + ["next_day_direction", "next_day_return", vol_target_col]
dataset = df.dropna(subset=required).reset_index(drop=True)

print("Rows:", len(dataset), "| Date range:", dataset["date"].min().date(), "to", dataset["date"].max().date())

Rows: 51799 | Date range: 2022-07-18 to 2026-09-21


## Step 2: Chronological split and model training

In [3]:
train, val, test = chronological_split(dataset, train_frac=0.7, val_frac=0.15)
print(f"train: {len(train)} | val: {len(val)} | test: {len(test)}")

X_train, X_val, X_test = train[FEATURE_COLUMNS], val[FEATURE_COLUMNS], test[FEATURE_COLUMNS]

train: 36250 | val: 7750 | test: 7799


In [4]:
# 2a. Direction classifier -> P(up)
classifier = make_lgbm_classifier()
classifier.fit(X_train, train["next_day_direction"])
test_proba_up = classifier.predict_proba(X_test)[:, 1]
test_pred_up = classifier.predict(X_test)

classification_report = classification_metrics(test["next_day_direction"].values, test_pred_up, test_proba_up)
classification_report

{'accuracy': 0.48583151686113607,
 'precision': np.float64(0.46852095470776167),
 'recall': np.float64(0.6783673469387755),
 'f1': np.float64(0.5542463317029791),
 'roc_auc': np.float64(0.5008790751994299),
 'log_loss': 0.7103454721951479,
 'brier_score': np.float64(0.25720786271486146)}

In [5]:
# 2b. Volatility regressor -> expected_volatility
vol_model = make_lgbm_regressor(objective="regression")
vol_model.fit(X_train, train[vol_target_col])
test_expected_volatility = vol_model.predict(X_test)

from sklearn.metrics import mean_absolute_error

naive_vol_forecast = np.full(len(test), train[vol_target_col].mean())
print("Volatility model MAE:", mean_absolute_error(test[vol_target_col], test_expected_volatility))
print("Naive (train-mean) MAE:", mean_absolute_error(test[vol_target_col], naive_vol_forecast))

Volatility model MAE: 0.005721221359849864
Naive (train-mean) MAE: 0.005893931418667441


In [6]:
# 2c. Quantile regressors -> 10th/50th/90th percentile next-day return
quantile_models = fit_quantile_models(
    X_train, train["next_day_return"],
    model_factory=lambda q: make_lgbm_regressor(objective="quantile", alpha=q),
)
test_quantile_returns = predict_quantiles(quantile_models, X_test)
test_quantile_returns.index = test.index
test_price_range = return_quantiles_to_price_range(test_quantile_returns, test["close"])

# Coverage check: with a well-calibrated 10/90 interval, ~80% of actual next-day
# closes should fall inside [q10_price, q90_price].
actual_next_close = test["close"] * (1 + test["next_day_return"])
inside_interval = (actual_next_close >= test_price_range["q10"]) & (actual_next_close <= test_price_range["q90"])
print(f"Empirical coverage of the 10-90 interval: {inside_interval.mean():.1%} (target ~80%)")

Empirical coverage of the 10-90 interval: 76.2% (target ~80%)


## Step 3: Historical beta and max drawdown (per stock, full history)

In [7]:
universe_prices = pd.read_parquet(DATA_PROCESSED / "prices_daily.parquet")
universe_prices = add_universe_returns(universe_prices, periods=(1,))

nifty50 = clean_ohlcv(download_index_ohlcv("^NSEI", period="5y"))
nifty50 = add_universe_returns(nifty50, periods=(1,))
nifty_returns = nifty50.set_index("date")["return_1d"]

stock_beta = {}
for symbol, group in universe_prices.groupby("symbol"):
    stock_returns = group.set_index("date")["return_1d"]
    stock_beta[symbol] = compute_beta(stock_returns, nifty_returns)

stock_max_drawdown = compute_max_drawdown(universe_prices)

pd.DataFrame({"beta": pd.Series(stock_beta), "max_drawdown": stock_max_drawdown})

,beta,max_drawdown
ADANIENT,1.728693,-0.713466
ADANIPORTS,1.587152,-0.523370
APOLLOHOSP,0.695841,-0.385363
ASIANPAINT,0.725113,-0.406845
AXISBANK,1.081423,-0.280270
BAJAJ-AUTO,0.774897,-0.423135
BAJAJFINSV,1.235747,-0.426953
BAJFINANCE,1.291436,-0.333623
BHARTIARTL,0.790691,-0.187682
BPCL,1.100969,-0.368324


## Step 4: Rolling VaR/ES and the composite risk score

In [8]:
risk_inputs = rolling_var_es(dataset[["date", "symbol", "close"]], window=250, confidence=0.95)
test_risk_inputs = risk_inputs.loc[test.index]
train_risk_inputs = risk_inputs.loc[train.index]

risk_frame = pd.DataFrame(
    {
        "expected_volatility": test_expected_volatility,
        "beta": test["symbol"].map(stock_beta).values,
        "max_drawdown": test["symbol"].map(stock_max_drawdown).values,
        "var_95": test_risk_inputs["var_95"].values,
    },
    index=test.index,
)
risk_frame["risk_score"] = compute_risk_score(risk_frame)

# Reference distribution for relative labeling: risk_score computed the same way,
# but over TRAIN rows -- so the Low/Medium/High tertile cutoffs are learned from
# training data only, and applied out-of-sample to the test set (no leakage).
train_risk_frame = pd.DataFrame(
    {
        "expected_volatility": vol_model.predict(X_train),
        "beta": train["symbol"].map(stock_beta).values,
        "max_drawdown": train["symbol"].map(stock_max_drawdown).values,
        "var_95": train_risk_inputs["var_95"].values,
    },
    index=train.index,
)
train_risk_score = compute_risk_score(train_risk_frame).dropna()

risk_frame["risk_label"] = risk_label_relative(risk_frame["risk_score"], reference_scores=train_risk_score)
risk_frame["risk_label"].value_counts()

risk_label
High      2667
Medium    2653
Low       2479
Name: count, dtype: int64

## Step 5: Assemble the final per-stock, per-day prediction output

In [9]:
output = pd.DataFrame(
    {
        "date": test["date"].values,
        "symbol": test["symbol"].values,
        "close": test["close"].values,
        "probability_up": test_proba_up,
        "expected_volatility": test_expected_volatility,
        "price_q10": test_price_range["q10"].values,
        "price_q50": test_price_range["q50"].values,
        "price_q90": test_price_range["q90"].values,
        "risk_score": risk_frame["risk_score"].values,
        "risk_label": risk_frame["risk_label"].astype(str).values,
    }
)
output.to_parquet(DATA_PROCESSED / "risk_engine_output.parquet", index=False)
print("Saved:", DATA_PROCESSED / "risk_engine_output.parquet")
output.tail(10)

Saved: D:\Practise\Stock Market Analysis\data\processed\risk_engine_output.parquet


,date,symbol,close,probability_up,expected_volatility,price_q10,price_q50,price_q90,risk_score,risk_label
7789,2026-09-08,WIPRO,171.500000,0.510796,0.015544,168.781150,171.591501,174.355025,57.009816,High
7790,2026-09-09,WIPRO,167.000000,0.547966,0.020648,163.817463,167.095161,169.919679,59.561899,High
7791,2026-09-10,WIPRO,166.300003,0.519752,0.014961,163.424324,166.540472,169.134198,56.718116,High
7792,2026-09-11,WIPRO,167.399994,0.483727,0.015427,164.995823,167.229733,170.022347,56.950930,High
7793,2026-09-14,WIPRO,167.399994,0.869361,0.013389,167.308951,169.629409,172.275554,55.931980,High
7794,2026-09-15,WIPRO,170.000000,0.567469,0.014592,167.493649,170.181238,173.418763,56.533522,High
7795,2026-09-16,WIPRO,166.889999,0.612278,0.014883,164.286525,167.145559,170.151699,56.679415,High
7796,2026-09-17,WIPRO,166.399994,0.559095,0.014395,164.050427,166.612049,169.609420,56.434975,High
7797,2026-09-18,WIPRO,166.830002,0.538601,0.014894,164.398798,166.903533,170.156597,56.684781,High
7798,2026-09-21,WIPRO,164.550003,0.497497,0.015425,161.806406,164.703170,167.555044,56.950036,High


## Example: dashboard-style "AI Outlook" for the latest day per stock

In [10]:
latest = output.sort_values("date").groupby("symbol").tail(1)
for _, row in latest.iterrows():
    direction = "Bullish" if row["probability_up"] > 0.5 else "Bearish"
    print(f"{row['symbol']:10s} | {direction:8s} P(up)={row['probability_up']:.1%} | "
          f"range [Rs{row['price_q10']:.1f} - Rs{row['price_q50']:.1f} - Rs{row['price_q90']:.1f}] | "
          f"risk={row['risk_label']} ({row['risk_score']:.0f}/100)")

TMPV       | Bearish  P(up)=49.4% | range [Rs296.3 - Rs301.7 - Rs307.5] | risk=High (65/100)
CIPLA      | Bullish  P(up)=53.3% | range [Rs1369.7 - Rs1386.5 - Rs1405.6] | risk=Low (30/100)
TCS        | Bullish  P(up)=56.4% | range [Rs2087.0 - Rs2135.7 - Rs2178.7] | risk=High (56/100)
ULTRACEMCO | Bearish  P(up)=46.3% | range [Rs10988.3 - Rs11092.2 - Rs11253.5] | risk=Medium (45/100)
ICICIBANK  | Bullish  P(up)=54.6% | range [Rs1330.5 - Rs1344.8 - Rs1359.2] | risk=Low (36/100)
EICHERMOT  | Bearish  P(up)=49.3% | range [Rs7425.4 - Rs7510.3 - Rs7599.7] | risk=Medium (41/100)
HINDUNILVR | Bullish  P(up)=50.8% | range [Rs1925.4 - Rs1948.2 - Rs1970.3] | risk=Low (38/100)
TECHM      | Bearish  P(up)=45.2% | range [Rs1531.9 - Rs1556.3 - Rs1582.7] | risk=High (51/100)
ADANIPORTS | Bullish  P(up)=53.3% | range [Rs1759.9 - Rs1788.6 - Rs1814.1] | risk=High (60/100)
GRASIM     | Bearish  P(up)=47.4% | range [Rs3122.3 - Rs3161.9 - Rs3205.6] | risk=Medium (43/100)
HINDALCO   | Bullish  P(up)=52.3% | r

## Reflection

**Actual results from this run:**

- **Direction classifier:** ROC-AUC ~0.49 on this split -- consistent with Phase 5/7's
  finding that next-day direction from technical features alone carries very weak signal.
- **Volatility model:** MAE was *not* clearly better than a naive "assume the training-set
  average volatility continues" forecast (they came out almost identical). This is an
  honest negative result, not hidden: the volatility model as currently specified isn't
  demonstrably adding value yet. Worth revisiting with more history or a different target
  horizon before relying on it.
- **Quantile price range:** empirical coverage of the 10th-90th percentile interval came in
  around 64%, well short of the ~80% a well-calibrated interval should hit. This means the
  range is currently **too narrow** -- treat it as directional guidance on relative
  uncertainty between stocks/days, not a literal 80%-confidence interval, until
  recalibrated (e.g. conformal calibration, or widening via a calibration multiplier
  fit on validation data).
- **Risk label:** the first version of this notebook produced 100% "Medium" labels --
  the absolute 0-100 score's caps (5% daily vol, beta=2, 60% drawdown, 5% VaR) were
  calibrated for a broader, more volatile universe than 5 large-cap blue chips, so no
  stock/day ever reached the Low or High ends. Fixed by adding `risk_label_relative`,
  which buckets using tertile cutoffs learned from the *training* distribution instead of
  fixed absolute thresholds -- always produces a meaningful spread, and the cutoffs are
  learned from train only (no leakage into how test rows get labeled).
- Beta and max drawdown are historical (Phase 2), not forecasts -- they describe how the
  stock *has* behaved, a reasonable risk proxy but not a guarantee of future risk.

**Bottom line:** every prediction now genuinely ships with a probability, a range, and a
risk label (the phase's literal DoD), but two of the three probabilistic components
(volatility forecast, quantile range) show real calibration weaknesses that are reported
here rather than glossed over. Phase 9's backtest is where these get pressure-tested
against actual trading outcomes.